# API Overlap Run

Готовый ноутбук для overlap-модели: обучение на USA train с пересекающимися признаками и directed OD-прогноз внутри выбранного региона РФ через `main_info` API.


In [10]:
# При первом запуске в новом окружении можно раскомментировать строку ниже.
# %pip install .


In [11]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import FileLink, display

PROJECT_ROOT = Path.cwd()
MIGRATION_DIR = PROJECT_ROOT / 'migration'

if str(MIGRATION_DIR) not in sys.path:
    sys.path.insert(0, str(MIGRATION_DIR))

import config
import api_overlap
from data_loading import load_datasets
from model_training import train_and_save_model

api_overlap = importlib.reload(api_overlap)
run_api_overlap_pipeline = api_overlap.run_api_overlap_pipeline

pd.options.display.max_columns = 100
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'API overlap model path: {config.API_OVERLAP_MODEL_PATH}')


PROJECT_ROOT: d:\programming\github\Migration-model
API overlap model path: models/RF/rf_model_api_overlap.joblib


In [12]:
TERRITORY_ID = 13517
DOWN_BY = 1
LOOP = True
LOOP_REQUEST_SLEEP_SECONDS = config.LOOP_REQUEST_SLEEP_SECONDS
TOP_PERCENT = 10
TOP_N = None

MODEL_PATH = PROJECT_ROOT / config.API_OVERLAP_MODEL_PATH
BASE_URL = config.MAIN_INFO_API_URL
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'api_overlap_notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RETRAIN_MODEL = False
RUN_SUFFIX = '_loop' if LOOP else ''

print(f'TERRITORY_ID = {TERRITORY_ID}')
print(f'DOWN_BY = {DOWN_BY}')
print(f'LOOP = {LOOP}')
print(f'LOOP_REQUEST_SLEEP_SECONDS = {LOOP_REQUEST_SLEEP_SECONDS}')
print(f'TOP_PERCENT = {TOP_PERCENT}')
print(f'TOP_N = {TOP_N}')
print(f'BASE_URL = {BASE_URL}')
print(f'MODEL_PATH = {MODEL_PATH}')
print(f'RETRAIN_MODEL = {RETRAIN_MODEL}')


TERRITORY_ID = 13517
DOWN_BY = 1
LOOP = True
LOOP_REQUEST_SLEEP_SECONDS = 0.1
TOP_PERCENT = 10
TOP_N = None
BASE_URL = http://10.32.1.47:5000/api/migrations/main_info
MODEL_PATH = d:\programming\github\Migration-model\models\RF\rf_model_api_overlap.joblib
RETRAIN_MODEL = False


In [13]:
if RETRAIN_MODEL or not MODEL_PATH.exists():
    train_data, test_data, _ = load_datasets(str(PROJECT_ROOT / 'data'), train_variant='full')
    _, evaluation_report = train_and_save_model(
        train_data=train_data,
        test_data=test_data,
        model_path=str(MODEL_PATH),
        feature_cols=config.API_OVERLAP_FEATURE_COLS,
    )
    display(evaluation_report)
else:
    print(f'Используется готовая overlap-модель: {MODEL_PATH}')


Используется готовая overlap-модель: d:\programming\github\Migration-model\models\RF\rf_model_api_overlap.joblib


In [14]:
result = run_api_overlap_pipeline(
    territory_id=TERRITORY_ID,
    down_by=DOWN_BY,
    loop=LOOP,
    loop_request_sleep_seconds=LOOP_REQUEST_SLEEP_SECONDS,
    top_percent=TOP_PERCENT,
    top_n=TOP_N,
    model_path=str(MODEL_PATH),
    base_url=BASE_URL,
    output_dir=OUTPUT_DIR / f'territory_{TERRITORY_ID}_down_{DOWN_BY}{RUN_SUFFIX}',
)

summary = pd.DataFrame([
    {
        'loop': result['loop'],
        'request_count': result['request_count'],
        'first_level_territories': len(result['first_level_ids']),
        'territories': len(result['territories']),
        'od_pairs': len(result['predictions']),
        'predicted_flow_sum': float(result['predictions']['total_pop_flow'].sum()),
        'predicted_flow_mean': float(result['predictions']['total_pop_flow'].mean()),
        'predicted_flow_max': float(result['predictions']['total_pop_flow'].max()),
    }
])
display(summary)
display(result['predictions'].sort_values('total_pop_flow', ascending=False).head(20))


Second-level API requests: 100%|██████████| 22/22 [02:05<00:00,  5.70s/it]
d:\programming\github\Migration-model\.venv\lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    7.4s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:   12.3s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:   15.6s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:   16.0s
[Parallel(n_jobs=12)]: Done 1000 out of 1000 | elapsed:   16.2s finished


,loop,request_count,first_level_territories,territories,od_pairs,predicted_flow_sum,predicted_flow_mean,predicted_flow_max
0,True,23,22,85,7140,719794.967454,100.81162,2718.238035


,d,area_o,area_d,m_o,m_d,health_point_o,health_point_d,main_road_line_o,main_road_line_d,school_point_o,school_point_d,territory_id_o,name_o,lat_o,lon_o,territory_id_d,name_d,lat_d,lon_d,total_pop_flow
3277,8.159909,1673.975691,92.210882,28087.0,38611.0,11.0,16.0,139.2,171.0,4.0,6.0,13596,Сельское поселение Советский,61.325674,63.581892,103440,город Югорск,61.313052,63.338133,2718.238035
6800,14.713939,805.151866,1578.797019,610.0,22292.0,2.0,2.0,15.6,21.1,1.0,5.0,13548,Сельское поселение Зайцева Речка,60.747199,76.675984,13550,Сельское поселение Излучинск,60.899871,76.978743,1785.948942
122,8.159909,92.210882,1673.975691,38611.0,28087.0,16.0,11.0,171.0,139.2,6.0,4.0,103440,город Югорск,61.313052,63.338133,13596,Сельское поселение Советский,61.325674,63.581892,1637.333024
1784,12.608519,738.563836,523.400035,4104.0,11423.0,3.0,2.0,60.5,81.3,2.0,1.0,13574,Сельское поселение Мортка,59.470479,66.236564,13573,Сельское поселение Междуреченский,59.586421,65.960032,1608.889793
6884,14.713939,1578.797019,805.151866,22292.0,610.0,2.0,2.0,21.1,15.6,5.0,1.0,13550,Сельское поселение Излучинск,60.899871,76.978743,13548,Сельское поселение Зайцева Речка,60.747199,76.675984,1537.276107
4938,13.648507,817.684611,944.475063,5176.0,763.0,3.0,1.0,33.3,16.9,3.0,1.0,13547,Сельское поселение Горноправдинск,60.007051,69.887712,13611,Сельское поселение Цингалы,60.193232,69.757852,1472.248539
7134,11.768249,287.573874,16.046282,632.0,473.0,1.0,1.0,9.1,7.5,1.0,1.0,13582,Сельское поселение Покур,61.008348,75.473935,13543,Сельское поселение Вата,61.087413,75.784253,1416.086712
5686,13.648507,944.475063,817.684611,763.0,5176.0,1.0,3.0,16.9,33.3,1.0,3.0,13611,Сельское поселение Цингалы,60.193232,69.757852,13547,Сельское поселение Горноправдинск,60.007051,69.887712,1331.042687
1700,12.608519,523.400035,738.563836,11423.0,4104.0,2.0,3.0,81.3,60.5,1.0,2.0,13573,Сельское поселение Междуреченский,59.586421,65.960032,13574,Сельское поселение Мортка,59.470479,66.236564,1327.850429
3830,11.351003,316.401525,210.071363,742.0,431.0,1.0,2.0,14.8,8.9,1.0,1.0,13569,Сельское поселение Лямина,61.270365,71.731237,13604,Сельское поселение Тундрино,61.236768,72.064348,1220.770815


In [15]:
display(FileLink(str(result['predictions_path'])))
display(FileLink(str(result['map_path'])))
result['territories'].head()


d:\programming\github\Migration-model\artifacts\api_overlap_notebook\territory_13517_down_1_loop\predictions_territory_13517_down_1_loop.csv

d:\programming\github\Migration-model\artifacts\api_overlap_notebook\territory_13517_down_1_loop\migration_map_territory_13517_down_1_loop.html

,territory_id,name,lat,lon,area,m,health_point,main_road_line,school_point,geometry
0,103554,тер 17 км автодороги Нефтеюганск-Тундрино,61.089521,72.619547,83.058299,1089.0,0.0,2.6,1.0,"POLYGON ((72.56864 61.08384, 72.56859 61.08395..."
1,103440,город Югорск,61.313052,63.338133,92.210882,38611.0,16.0,171.0,6.0,"MULTIPOLYGON (((63.15194 61.27881, 63.15219 61..."
2,13545,Сельское поселение Верхнеказымский,63.818963,67.793183,788.451813,1868.0,1.0,0.0,1.0,"POLYGON ((67.6358 63.7685, 67.63583 63.76841, ..."
3,13551,Сельское поселение Казым,63.644158,68.963264,8420.186755,1600.0,3.0,0.0,1.0,"POLYGON ((67.08482 63.73113, 67.09698 63.61671..."
4,13568,Сельское поселение Лыхма,63.210106,66.918646,597.891610,1505.0,1.0,0.0,1.0,"POLYGON ((66.83468 63.27549, 66.83525 63.24207..."


In [16]:
from visualization import create_migration_graph

preview_map = create_migration_graph(
    result['predictions'],
    result['territories'],
    top_percent=4,
    top_n=TOP_N,
    node_o='territory_id_o',
    node_d='territory_id_d',
    coord_name='territory_id',
    tooltip_cols=['territory_id_o', 'name_o', 'territory_id_d', 'name_d', 'd', 'total_pop_flow'],
    polygon_gdf=result['territories'],
    polygon_id_col='territory_id',
    polygon_name_col='name',
    polygon_tooltip_cols=['name', 'predicted_outflow', 'predicted_inflow', 'predicted_balance'],
)
preview_map.save(f'{TERRITORY_ID}{RUN_SUFFIX}.html')
